# 02 — Mendeley Dataset Preprocessing

This notebook prepares the Mendeley Heart Disease dataset for machine learning by cleaning the data, handling invalid values, splitting it into training and testing sets, applying feature scaling, and exporting reusable datasets and preprocessing artifacts for subsequent notebooks.

In [16]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

In [2]:
df = pd.read_csv("../data/raw/mendeley_heart_disease.csv")

df.head()

,patientid,age,gender,chestpain,restingBP,serumcholestrol,fastingbloodsugar,restingrelectro,maxheartrate,exerciseangia,oldpeak,slope,noofmajorvessels,target
0,103368,53,1,2,171,0,0,1,147,0,5.3,3,3,1
1,119250,40,1,0,94,229,0,1,115,0,3.7,1,1,0
2,119372,49,1,2,133,142,0,0,202,1,5.0,1,0,0
3,132514,43,1,0,138,295,1,1,153,0,3.2,2,2,1
4,146211,31,1,1,199,0,0,2,136,0,5.3,3,2,1


In [3]:
print("Shape Before Cleaning :", df.shape)

Shape Before Cleaning : (1000, 14)


## Removal of Non-Predictive Features

In [4]:
df.drop(columns=["patientid"], inplace=True)

df.head()

,age,gender,chestpain,restingBP,serumcholestrol,fastingbloodsugar,restingrelectro,maxheartrate,exerciseangia,oldpeak,slope,noofmajorvessels,target
0,53,1,2,171,0,0,1,147,0,5.3,3,3,1
1,40,1,0,94,229,0,1,115,0,3.7,1,1,0
2,49,1,2,133,142,0,0,202,1,5.0,1,0,0
3,43,1,0,138,295,1,1,153,0,3.2,2,2,1
4,31,1,1,199,0,0,2,136,0,5.3,3,2,1


Patient identifiers do not contain predictive clinical information and may introduce unnecessary noise into the modelling process. Therefore, they are removed before training machine learning models.

## Handling Invalid Clinical Measurements

In [5]:
(df["serumcholestrol"] == 0).sum()

np.int64(53)

In [6]:
median_chol = df.loc[
    df["serumcholestrol"] != 0,
    "serumcholestrol"
].median()

df["serumcholestrol"] = df["serumcholestrol"].replace(
    0,
    median_chol
)

Exploratory analysis identified several serum cholesterol values equal to zero, which are medically implausible. These values are treated as invalid measurements and replaced using the median of valid cholesterol observations to preserve the overall distribution while minimizing the influence of outliers.

## Feature and Target Separation

In [7]:
X = df.drop("target", axis=1)

y = df["target"]

In [8]:
print(X.shape)
print(y.shape)

(1000, 12)
(1000,)


## Train-Test Split

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Stratified sampling is employed to preserve the class distribution in both training and testing datasets, ensuring reliable and unbiased model evaluation.

## Feature Scaling

In [10]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [11]:
X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=X_train.columns
)

X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=X_test.columns
)

Feature scaling is performed using standardization to ensure that variables measured on different scales contribute equally during model training. This step is particularly important for distance-based and optimization-based algorithms such as KNN, SVM, and Logistic Regression.

## Export Processed Dataset

In [14]:
# Save the scaled feature dataframes and targets
X_train_scaled.to_csv(
    "../data/processed/X_train_scaled.csv",
    index=False
)

X_test_scaled.to_csv(
    "../data/processed/X_test_scaled.csv",
    index=False
)

# Targets are typically categorical for this dataset; save without scaling
pd.DataFrame(y_train).to_csv(
    "../data/processed/y_train.csv",
    index=False
)

pd.DataFrame(y_test).to_csv(
    "../data/processed/y_test.csv",
    index=False
)

Preprocessing Summary

• Non-predictive identifier columns were removed.
• Medically invalid cholesterol values were corrected using median imputation.
• Features and target variables were separated for modelling.
• Stratified train-test splitting was applied to preserve class balance.
• Numerical features were standardized to improve model performance.
• The cleaned dataset was exported for subsequent machine learning experiments.

In [17]:
joblib.dump(scaler, "../saved_models/scaler.pkl")

['../saved_models/scaler.pkl']

## Conclusion

The dataset has been cleaned, split into training and testing sets, standardized using a fitted scaler, and exported for downstream tasks. The processed datasets and preprocessing artifacts generated in this notebook will be used for model training and evaluation in the next notebook.